In [1]:
import pandas as pd
import numpy as np 
from collections import defaultdict



In [2]:
df = pd.read_csv(r"C:\Users\amalm\OneDrive\Desktop\finamcial_data_analysis_lerning\project\Risk Scoring Engine\data\sample_transaction_data.csv")
df.head()

,user_id,amount,balance,timestamp,txn_type,merchant,location,channel,device_id,txn_status
0,user_1,4315.28,64509.78,2025-06-09 08:52:24,credit,Wilson-Welch,East Maryhaven,ATM,device_37,success
1,user_1,702.12,45006.57,2025-06-09 15:45:18,debit,Johnson LLC,West Emma,POS,device_608,success
2,user_1,7544.32,16609.86,2025-06-10 00:08:28,debit,Orozco PLC,Joseside,POS,device_657,success
3,user_1,8683.85,23572.25,2025-06-10 00:55:36,debit,Anderson-Smith,West Stephanieton,ATM,device_407,fail
4,user_1,3604.73,31008.53,2025-06-10 02:13:14,credit,Molina-Gill,Kristinahaven,POS,device_140,success


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 97067 entries, 0 to 97066
Data columns (total 10 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   user_id     97067 non-null  object 
 1   amount      97067 non-null  float64
 2   balance     97067 non-null  float64
 3   timestamp   97067 non-null  object 
 4   txn_type    97067 non-null  object 
 5   merchant    97067 non-null  object 
 6   location    97067 non-null  object 
 7   channel     97067 non-null  object 
 8   device_id   97067 non-null  object 
 9   txn_status  97067 non-null  object 
dtypes: float64(2), object(8)
memory usage: 7.4+ MB


In [4]:
# Changing  timestamp data_type to datetime
df['timestamp'] = pd.to_datetime(df['timestamp'])
df = df.sort_values(['user_id','timestamp']).reset_index(drop=True)


# Fraud Risk Scoring Rules 
This scoring engine assigns a fraud risk score to each user based on their transaction behavior. The system uses defined, logic-based rules. Each rule violation adds +2 risk points.

 ### Rule 1: Frequent Low Balance
Objective: Detect users who often maintain a very low account balance (a common pattern in fraud, money laundering layering, or account misuse).

Logic:
If a user’s balance drops below ₹100 more than 5 times within the observed time window (e.g., 30 days), they are flagged.

Score Contribution: +2 risk points

 Ethical Consideration:
 
This rule may flag low-income users who are not fraudulent. In reality, frequent low balances are common in economically stressed groups. This highlights the limitations of rule-based systems and the need for context-aware scoring in real-world applications.



### Rule 2: High Transaction Frequency in Short Time
Objective: Identify burst patterns of transactions, often seen in bot-based fraud or smurfing.

Logic:
If a user performs more than 5 transactions within a 10-minute window, they are flagged.

Score Contribution: +2 risk points

While high-frequency transactions can be a sign of fraud, it’s also very normal in UPI-heavy environments like India. That’s why I treat this as a supporting rule, not a standalone red flag.
In production systems, we’d combine this with device ID changes, geo-locations, or abnormal amounts to reduce false positives

### Rule 3: Abnormally Large Transactions
Objective: Spot outliers in transaction amounts that don't match the user’s usual financial behavior.

Logic:
If a user makes any transaction that is more than 3 times their average transaction amount, they are flagged.

Score Contribution: +2 risk points

## Total Risk Score & Risk Category

Each user starts with a score of 0.

Points from triggered rules are added.

The final fraud risk level is categorized as:

Risk Score	Risk Level
0–2	Low
3–4	Medium
5+ High

## Rule 1: Frequent Low Balance (< ₹100 more than 5 times)



In [11]:
def rule_frequent_low_balance(df, threshold=100, count_limit=5):
    low_balance = df[df['balance'] < threshold]
    
    # Counting how many times each user had low balance
    user_low_count = low_balance['user_id'].value_counts()
    
    # Filtering users who triggered the rule
    flagged_users = user_low_count[user_low_count > count_limit].index.tolist()
    
    return flagged_users

    

## Rule 2: High Transaction Frequency in Short Time

In [6]:
def rule_high_freq(df, txn_limit=3, time_window='30min'):
    flagged_users = []
    df_sorted = df.sort_values(by=['user_id', 'timestamp'])
    
    for user, group in df_sorted.groupby('user_id'):
        group = group.reset_index(drop=True)
        group['timestamp'] = pd.to_datetime(group['timestamp'])
        
        for i in range(len(group) - txn_limit + 1):
            window = group.loc[i:i+txn_limit-1]
            time_diff = window['timestamp'].iloc[-1] - window['timestamp'].iloc[0]
            if time_diff <= pd.Timedelta(time_window):
                flagged_users.append(user)
                break
                
    return flagged_users


## Rule 3: Abnormally Large Transaction

In [7]:
def rule_abnormal_txn(df, multiplier=2):
    flagged_users = []
    for user, group in df.groupby('user_id'):
        avg_amount = group['amount'].mean()
        if any(group['amount'] > multiplier * avg_amount):
            flagged_users.append(user)
    return flagged_users


In [8]:


# Initialize scores
risk_scores = defaultdict(int)

# Add scores for each rule
for user in rule_frequent_low_balance(df):
    risk_scores[user] += 2
for user in rule_high_freq(df):
    risk_scores[user] += 2
for user in rule_abnormal_txn(df):
    risk_scores[user] += 2

# Create DataFrame from scores
risk_df = pd.DataFrame([
    {'user_id': user, 'risk_score': score} 
    for user, score in risk_scores.items()
])

# Assign risk level
def risk_label(score):
    if score <= 2:
        return "Low"
    elif score <= 4:
        return "Medium"
    else:
        return "High"

risk_df['risk_level'] = risk_df['risk_score'].apply(risk_label)


In [12]:
# Merge Back to Original Transaction Data
df_final = df.merge(risk_df, on='user_id', how='left')
df_final['risk_level'] = df_final['risk_level'].fillna("Low")
df_final.to_csv("transactions_with_risk.csv", index=False)



In [13]:
risk_df.to_csv(r"C:\Users\amalm\OneDrive\Desktop\finamcial_data_analysis_lerning\project\Risk Scoring Engine\data\fraud_risk_scores_new.csv", index=False)
